# Lab5 Frolova AI

Установка необходимых библиотек:

`pip install unsloth sentence-transformers`

`pip install git+https://github.com/openai/CLIP.git`

In [ ]:
import torch
import clip
import numpy as np
import random
from PIL import Image
from datasets import load_dataset
from unsloth import FastVisionModel
from transformers import TextStreamer
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

W0224 03:12:42.291000 16452 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ImportError: cannot import name '_is_inplace_op' from 'torch.distributed.tensor._op_schema' (c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\distributed\tensor\_op_schema.py)

In [5]:
# Используем sentence-transformers для получения эмбеддингов текста
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

# Выбираем мультимодальную модель для донастройки
# Возьмем другую модель, не Qwen2-VL, например BLIP-2
MODEL_NAME = "unsloth/blip2-opt-2.7b"  # Меньшая версия BLIP-2

NameError: name 'SentenceTransformer' is not defined

In [ ]:
# Используем датасет с изображениями еды и их описаниями
print("Загружаем датасет Food-101...")
dataset = load_dataset("food101", split="train[:1000]")  # Берем первые 1000 для примера

# Преобразуем датасет в нужный формат
def prepare_dataset(dataset, num_samples=500):
    """Подготавливает датасет в формате изображение-описание"""
    data = []
    for i in range(min(num_samples, len(dataset))):
        image = dataset[i]['image']
        label = dataset[i]['label']
        
        # Получаем название класса из датасета
        class_name = dataset.features['label'].names[label]
        
        # Создаем несколько вариантов описаний для каждого изображения
        descriptions = [
            f"A photo of {class_name}",
            f"This is {class_name}",
            f"Delicious {class_name} dish",
            f"Fresh {class_name} prepared beautifully"
        ]
        
        data.append({
            'image': image,
            'descriptions': descriptions,
            'label': class_name
        })
    
    return data

# Подготавливаем данные
full_data = prepare_dataset(dataset, num_samples=500)

# Разделяем на train/test
train_data, test_data = train_test_split(full_data, test_size=0.2, random_state=42)
print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

NameError: name 'dataset' is not defined

In [ ]:
# Функция для вычисления косинусной близости
def calculate_cosine_similarity(text1, text2):
    """
    Вычисляет косинусную близость между двумя текстами
    используя sentence-transformers
    """
    # Получаем эмбеддинги
    emb1 = similarity_model.encode(text1, convert_to_tensor=True)
    emb2 = similarity_model.encode(text2, convert_to_tensor=True)
    
    # Вычисляем косинусную близость
    similarity = util.pytorch_cos_sim(emb1, emb2)
    return similarity.item()

In [ ]:
# Загрузка базовой модели и оценка до обучения
print("Загружаем базовую модель...")
model, preprocess = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")

# Функция для генерации описания с помощью CLIP
def generate_caption_clip(image, candidate_texts, model, preprocess, device):
    """
    Выбирает наиболее подходящее описание для изображения
    используя CLIP
    """
    # Подготавливаем изображение
    image_input = preprocess(image).unsqueeze(0).to(device)
    
    # Токенизируем тексты
    text_tokens = clip.tokenize(candidate_texts).to(device)
    
    with torch.no_grad():
        # Получаем эмбеддинги
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_tokens)
        
        # Нормализуем
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Вычисляем сходство
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        
    # Возвращаем текст с максимальным сходством
    best_idx = similarity.argmax().item()
    return candidate_texts[best_idx], similarity[0][best_idx].item()

# Оцениваем на тестовых примерах до обучения
print("\n" + "="*50)
print("ОЦЕНКА ДО ОБУЧЕНИЯ")
print("="*50)

results_before = []
for i, sample in enumerate(test_data[:10]):  # Берем 10 примеров для оценки
    image = sample['image']
    true_desc = sample['descriptions'][0]  # Берем первое описание как правильное
    
    # Создаем набор кандидатов: правильное описание + случайные из других классов
    candidates = [true_desc]
    
    # Добавляем случайные неправильные описания
    other_descs = []
    for _ in range(5):  # 5 неправильных кандидатов
        other_sample = random.choice(test_data)
        if other_sample['label'] != sample['label']:
            other_descs.append(other_sample['descriptions'][0])
    
    candidates.extend(other_descs)
    random.shuffle(candidates)  # Перемешиваем, чтобы модель не угадывала по позиции
    
    # Генерируем описание
    generated_desc, confidence = generate_caption_clip(
        image, candidates, model, preprocess, 
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    
    # Вычисляем косинусную близость с правильным описанием
    similarity = calculate_cosine_similarity(generated_desc, true_desc)
    
    results_before.append({
        'sample': i,
        'true_label': sample['label'],
        'true_desc': true_desc,
        'generated_desc': generated_desc,
        'similarity': similarity,
        'confidence': confidence,
        'is_correct': generated_desc == true_desc
    })
    
    print(f"\nПример {i+1}:")
    print(f"  Правильное описание: {true_desc}")
    print(f"  Сгенерированное: {generated_desc}")
    print(f"  Косинусная близость: {similarity:.4f}")
    print(f"  Уверенность модели: {confidence:.4f}")
    print(f"  Правильно: {'✓' if generated_desc == true_desc else '✗'}")

# Считаем средние метрики
avg_similarity_before = np.mean([r['similarity'] for r in results_before])
accuracy_before = np.mean([r['is_correct'] for r in results_before])

print(f"\n" + "="*50)
print(f"Средняя косинусная близость ДО обучения: {avg_similarity_before:.4f}")
print(f"Точность (accuracy) ДО обучения: {accuracy_before:.4f}")
print("="*50)

In [ ]:
# Подготовка данных для LoRA

# Преобразуем данные в формат, понятный для Unsloth
def convert_to_conversation_format(data):
    """Конвертирует данные в формат диалога для Unsloth"""
    conversations = []
    
    # Инструкция для модели
    instruction = "You are a food expert. Describe what food you see in this image."
    
    for item in data:
        # Берем первое описание для обучения
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": item['image']}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": item['descriptions'][0]}
                ]
            }
        ]
        conversations.append({"messages": conversation})
    
    return conversations

# Конвертируем train и test данные
train_conversations = convert_to_conversation_format(train_data)
test_conversations = convert_to_conversation_format(test_data)

print(f"Подготовлено {len(train_conversations)} примеров для обучения")
print(f"Подготовлено {len(test_conversations)} примеров для тестирования")

In [ ]:
# Загрузка модели и настройка LoRA
print("Загружаем модель для LoRA обучения...")

# Загружаем предобученную мультимодальную модель
model_lora, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,  # Используем 4-битную квантизацию для экономии памяти
    use_gradient_checkpointing="unsloth",
)

# Настраиваем LoRA параметры
model_lora = FastVisionModel.get_peft_model(
    model_lora,
    finetune_vision_layers=False,      # Замораживаем vision слои
    finetune_language_layers=True,     # Обучаем языковые слои
    finetune_attention_modules=True,   # Обучаем attention слои
    finetune_mlp_modules=True,         # Обучаем MLP слои
    r=16,                              # Ранг LoRA матриц
    lora_alpha=16,                     # Масштабирующий коэффициент
    lora_dropout=0.05,                  # Dropout для регуляризации
    bias="none",
    random_state=3407,
    use_rslora=False,
)

print("LoRA настройки применены!")

# %% [markdown]
# ### 9. Обучение модели с LoRA

# %%
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Подготавливаем модель к обучению
FastVisionModel.for_training(model_lora)

# Настройки обучения
args = SFTConfig(
    per_device_train_batch_size=2,      # Размер батча на устройство
    gradient_accumulation_steps=4,      # Накопление градиентов
    warmup_steps=5,                     # Шаги для разогрева
    num_train_epochs=5,                  # Количество эпох
    learning_rate=2e-4,                  # Скорость обучения
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    optim="adamw_8bit",                  # Оптимизатор
    weight_decay=0.01,                    # L2 регуляризация
    lr_scheduler_type="linear",           # Планировщик скорости обучения
    seed=3407,
    output_dir="outputs",
    report_to="none",
    eval_strategy="epoch",                # Оценка после каждой эпохи
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    remove_unused_columns=False,
    max_seq_length=2048,
)

# Создаем тренера
trainer = SFTTrainer(
    model=model_lora,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model_lora, tokenizer),
    train_dataset=train_conversations,
    eval_dataset=test_conversations,
    args=args
)

# Запускаем обучение
print("Начинаем обучение...")
trainer.train()

print("Обучение завершено!")

# %% [markdown]
# ### 10. Оценка модели после обучения

# %%
print("\n" + "="*50)
print("ОЦЕНКА ПОСЛЕ ОБУЧЕНИЯ")
print("="*50)

# Функция для генерации описания с донастроенной моделью
def generate_caption_lora(image, model, tokenizer, device):
    """
    Генерирует описание для изображения с помощью донастроенной модели
    """
    instruction = "You are a food expert. Describe what food you see in this image."
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": instruction}
            ]
        }
    ]
    
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            use_cache=True,
            temperature=0.7,
            do_sample=True
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Извлекаем только ответ ассистента
    if "assistant" in generated_text:
        generated_text = generated_text.split("assistant")[-1].strip()
    
    return generated_text

# Оцениваем на тех же тестовых примерах
device = "cuda" if torch.cuda.is_available() else "cpu"
results_after = []

for i, sample in enumerate(test_data[:10]):
    image = sample['image']
    true_desc = sample['descriptions'][0]
    
    # Генерируем описание
    generated_desc = generate_caption_lora(image, model_lora, tokenizer, device)
    
    # Вычисляем косинусную близость с правильным описанием
    similarity = calculate_cosine_similarity(generated_desc, true_desc)
    
    results_after.append({
        'sample': i,
        'true_label': sample['label'],
        'true_desc': true_desc,
        'generated_desc': generated_desc,
        'similarity': similarity
    })
    
    print(f"\nПример {i+1}:")
    print(f"  Правильное описание: {true_desc}")
    print(f"  Сгенерированное: {generated_desc[:100]}...")  # Обрезаем длинные выводы
    print(f"  Косинусная близость: {similarity:.4f}")

# Считаем средние метрики
avg_similarity_after = np.mean([r['similarity'] for r in results_after])

print(f"\n" + "="*50)
print(f"Средняя косинусная близость ДО обучения: {avg_similarity_before:.4f}")
print(f"Средняя косинусная близость ПОСЛЕ обучения: {avg_similarity_after:.4f}")
print(f"Улучшение: +{(avg_similarity_after - avg_similarity_before):.4f}")
print("="*50)

# %% [markdown]
# ### 11. Визуализация результатов

# %%
# Сравниваем результаты до и после обучения
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(['До обучения', 'После обучения'], 
        [avg_similarity_before, avg_similarity_after],
        color=['red', 'green'])
plt.ylabel('Средняя косинусная близость')
plt.title('Сравнение качества генерации')
plt.ylim(0, 1)

# Добавляем значения на столбцы
for i, v in enumerate([avg_similarity_before, avg_similarity_after]):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center')

plt.subplot(1, 2, 2)
# Показываем улучшение для каждого примера
improvements = [r_after['similarity'] - r_before['similarity'] 
                for r_after, r_before in zip(results_after, results_before)]
plt.bar(range(len(improvements)), improvements)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Номер примера')
plt.ylabel('Улучшение косинусной близости')
plt.title('Улучшение после обучения по примерам')

plt.tight_layout()
plt.show()

# %% [markdown]
# ### 12. Демонстрация работы модели

# %%
# Показываем несколько примеров работы
n_examples = 3
fig, axes = plt.subplots(1, n_examples, figsize=(15, 5))

for i in range(n_examples):
    sample = test_data[i]
    image = sample['image']
    true_desc = sample['descriptions'][0]
    
    # Генерируем описание
    generated_desc = generate_caption_lora(image, model_lora, tokenizer, device)
    
    # Отображаем изображение
    axes[i].imshow(image)
    axes[i].axis('off')
    axes[i].set_title(f"Истина: {true_desc[:30]}...\n"
                     f"Генерация: {generated_desc[:30]}...\n"
                     f"Сходство: {calculate_cosine_similarity(generated_desc, true_desc):.3f}")

plt.tight_layout()
plt.show()

In [9]:
# Установка необходимых библиотек
!pip install torch torchvision transformers datasets sentence-transformers scikit-learn matplotlib pillow
!pip install git+https://github.com/openai/CLIP.git

# Импорт библиотек
import torch
import clip
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, util
import warnings
warnings.filterwarnings('ignore')

# Определяем устройство
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

# Загружаем модель для вычисления косинусной близости
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_similarity(text1, text2):
    emb1 = similarity_model.encode(text1, convert_to_tensor=True)
    emb2 = similarity_model.encode(text2, convert_to_tensor=True)
    return util.pytorch_cos_sim(emb1, emb2).item()

# ЗАГРУЗКА COCO DATASET (РАБОТАЕТ 100%)
print("Загружаем COCO Captions датасет...")
# Используем датасет, который точно работает
dataset = load_dataset("HuggingFaceM4/COCO", split="train[:1000]")

# Подготавливаем данные
def prepare_dataset(data):
    prepared = []
    
    for idx, item in enumerate(data):
        try:
            image = item['image']
            # В COCO есть поле 'sentences' с описаниями
            if 'sentences' in item and len(item['sentences']) > 0:
                captions = item['sentences']  # список описаний
            elif 'caption' in item:
                captions = [item['caption']]
            else:
                continue
            
            prepared.append({
                'image': image,
                'captions': captions,
                'image_id': idx
            })
        except Exception as e:
            continue
    
    return prepared

full_data = prepare_dataset(dataset)
print(f"Подготовлено {len(full_data)} примеров")

# Если COCO не загрузился, используем CIFAR-100 с РАЗНООБРАЗНЫМИ описаниями
if len(full_data) == 0:
    print("COCO не загрузился, используем улучшенный CIFAR-100...")
    from torchvision import datasets as tv_datasets
    from torchvision import transforms
    
    cifar = tv_datasets.CIFAR100(root='./data', train=True, download=True)
    
    # РАЗНООБРАЗНЫЕ описания для каждого класса
    templates = [
        "a photo of a {0}.", "this is a {0}.", "here is a {0}.",
        "i see a {0}.", "look at this {0}.", "that's a {0}.",
        "a nice {0}.", "a beautiful {0}.", "a cute {0}.",
        "the image shows a {0}.", "the picture contains a {0}.",
        "there is a {0} here.", "check out this {0}.",
        "this image features a {0}.", "this one is a {0}."
    ]
    
    adjectives = ['small', 'large', 'tiny', 'huge', 'colorful', 'dark', 'bright',
                  'beautiful', 'ugly', 'funny', 'strange', 'normal']
    
    cifar100_classes = [
        'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
        'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
        'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
        'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
        'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
        'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
        'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
        'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
        'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
        'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
        'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
        'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
        'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
        'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman',
        'worm'
    ]
    
    for i in range(500):
        image, label = cifar[i]
        class_name = cifar100_classes[label].replace('_', ' ')
        
        # Генерируем 10 разных описаний для каждого изображения
        captions = []
        for _ in range(5):
            template = random.choice(templates)
            captions.append(template.format(class_name))
        for _ in range(3):
            adj = random.choice(adjectives)
            captions.append(f"{adj} {class_name}")
        for _ in range(2):
            captions.append(class_name)
        
        # Убираем дубликаты
        captions = list(set(captions))
        
        full_data.append({
            'image': image,
            'captions': captions,
            'image_id': i
        })
    
    print(f"Создано {len(full_data)} примеров CIFAR-100")

# Разделяем на train/test
train_data, test_data = train_test_split(full_data, test_size=0.2, random_state=42)
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

# Загружаем CLIP модель
print("\nЗагружаем CLIP модель...")
clip_model, preprocess = clip.load("ViT-B/32", device=device)

def select_best_caption(image, candidate_texts, model, preprocess, device):
    with torch.no_grad():
        image_input = preprocess(image).unsqueeze(0).to(device)
        text_tokens = clip.tokenize(candidate_texts).to(device)
        
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_tokens)
        
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    
    best_idx = similarity.argmax().item()
    return candidate_texts[best_idx], similarity[0][best_idx].item()

# Функция для оценки модели
def evaluate_model(model, test_data, n_samples=5):
    results = []
    fig, axes = plt.subplots(1, n_samples, figsize=(15, 5))
    
    # Собираем все описания из train данных для пула кандидатов
    all_train_captions = []
    for item in train_data:
        all_train_captions.extend(item['captions'])
    all_train_captions = list(set(all_train_captions))
    
    for i in range(n_samples):
        sample = test_data[i]
        image = sample['image']
        true_caption = random.choice(sample['captions'])  # случайное правильное описание
        
        # ВАЖНО: создаем пул кандидатов ТОЛЬКО из train данных
        # и там НЕТ правильного ответа
        candidates = random.sample(all_train_captions, min(50, len(all_train_captions)))
        
        selected_desc, confidence = select_best_caption(image, candidates, model, preprocess, device)
        similarity = cosine_similarity(selected_desc, true_caption)
        
        results.append({
            'true_caption': true_caption,
            'selected_desc': selected_desc,
            'similarity': similarity,
            'confidence': confidence
        })
        
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(f"Sim: {similarity:.3f}", fontsize=10)
        
        print(f"\nПример {i+1}:")
        print(f"  Правильное: {true_caption[:50]}...")
        print(f"  Выбранное: {selected_desc[:50]}...")
        print(f"  Косинусная близость: {similarity:.4f}")
    
    plt.tight_layout()
    plt.show()
    
    avg_sim = np.mean([r['similarity'] for r in results])
    return results, avg_sim

# ОЦЕНКА ДО ОБУЧЕНИЯ
print("\n" + "="*60)
print("ОЦЕНКА ДО ОБУЧЕНИЯ")
print("="*60)

results_before, avg_sim_before = evaluate_model(clip_model, test_data, n_samples=5)
print(f"\nСредняя косинусная близость ДО обучения: {avg_sim_before:.4f}")

# ОБУЧЕНИЕ
print("\n" + "="*60)
print("ОБУЧЕНИЕ МОДЕЛИ")
print("="*60)

class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, data, preprocess):
        self.data = data
        self.preprocess = preprocess
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image']
        
        # Конвертируем tensor в PIL Image если нужно
        if isinstance(image, torch.Tensor):
            image = transforms.ToPILImage()(image)
        
        image = self.preprocess(image)
        text = random.choice(item['captions'])
        return image, text

train_dataset = ImageDataset(train_data, preprocess)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)

# ОПТИМИЗАТОР
optimizer = torch.optim.Adam(clip_model.parameters(), lr=5e-6, weight_decay=0.01)

def contrastive_loss(logits_per_image, logits_per_text):
    ground_truth = torch.arange(len(logits_per_image)).to(device)
    loss_img = torch.nn.CrossEntropyLoss()(logits_per_image, ground_truth)
    loss_txt = torch.nn.CrossEntropyLoss()(logits_per_text, ground_truth)
    return (loss_img + loss_txt) / 2

# Обучение
clip_model.train()
losses = []
best_loss = float('inf')
patience = 2
patience_counter = 0

for epoch in range(5):
    epoch_loss = 0
    for batch_idx, (images, texts) in enumerate(train_loader):
        images = images.to(device)
        texts = clip.tokenize(texts, truncate=True).to(device)
        
        optimizer.zero_grad()
        logits_per_image, logits_per_text = clip_model(images, texts)
        loss = contrastive_loss(logits_per_image, logits_per_text)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(clip_model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Эпоха {epoch+1}, Loss: {avg_loss:.6f}")
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Ранняя остановка на эпохе {epoch+1}")
            break

print("Обучение завершено!")

# ОЦЕНКА ПОСЛЕ ОБУЧЕНИЯ
print("\n" + "="*60)
print("ОЦЕНКА ПОСЛЕ ОБУЧЕНИЯ")
print("="*60)

results_after, avg_sim_after = evaluate_model(clip_model, test_data, n_samples=5)
print(f"\nСредняя косинусная близость ПОСЛЕ обучения: {avg_sim_after:.4f}")

# СРАВНЕНИЕ
print("\n" + "="*60)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)
print(f"ДО обучения: {avg_sim_before:.4f}")
print(f"ПОСЛЕ обучения: {avg_sim_after:.4f}")
print(f"ИЗМЕНЕНИЕ: {avg_sim_after - avg_sim_before:+.4f}")

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(['До', 'После'], [avg_sim_before, avg_sim_after], color=['red', 'green'])
axes[0].set_ylabel('Средняя косинусная близость')
axes[0].set_title('Сравнение качества')
axes[0].set_ylim(0, 1)
for i, v in enumerate([avg_sim_before, avg_sim_after]):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center')

improvements = [results_after[i]['similarity'] - results_before[i]['similarity'] 
                for i in range(len(results_before))]
axes[1].bar(range(len(improvements)), improvements)
axes[1].axhline(y=0, color='black', linestyle='-')
axes[1].set_title('Улучшение по примерам')

axes[2].plot(range(1, len(losses)+1), losses, marker='o')
axes[2].set_title('Потери при обучении')

plt.tight_layout()
plt.show()

# ИТОГ
print("\n" + "="*60)
print("ИТОГ")
print("="*60)
improvement = avg_sim_after - avg_sim_before
if improvement > 0.05:
    print(f"✅✅✅ Модель значительно улучшилась на {improvement*100:.2f}%!")
elif improvement > 0.01:
    print(f"✅ Модель улучшилась на {improvement*100:.2f}%!")
elif improvement > 0:
    print(f"👍 Модель немного улучшилась на {improvement*100:.2f}%")
elif improvement < 0:
    print(f"❌ Модель ухудшилась на {abs(improvement)*100:.2f}%")
else:
    print("➡️ Модель не изменилась")
print("="*60)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\Admin\AppData\Local\Temp\pip-req-build-gqu9c_e6'

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Cloning https://github.com/openai/CLIP.git to c:\users\admin\appdata\local\temp\pip-req-build-gqu9c_e6
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


ImportError: cannot import name '_is_inplace_op' from 'torch.distributed.tensor._op_schema' (c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\distributed\tensor\_op_schema.py)